# 🤖 04 — Entraînement & Évaluation de 5 modèles
Comparaison Linear Regression, Ridge, Random Forest, Gradient Boosting, XGBoost.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd, numpy as np, warnings; warnings.filterwarnings('ignore')

session = get_active_session()
session.sql("USE DATABASE HOUSE_PRICE_DB").collect()

FEATURE_COLS = [
    'AREA','BEDROOMS','BATHROOMS','STORIES','MAINROAD','GUESTROOM',
    'BASEMENT','HOTWATERHEATING','AIRCONDITIONING','PARKING','PREFAREA',
    'FURNISHING_ENCODED','AREA_BEDS','TOTAL_ROOMS','PREMIUM_FLAG'
]

train = session.table("ML.TRAIN_SET").to_pandas()
test = session.table("ML.TEST_SET").to_pandas()

X_train, y_train = train[FEATURE_COLS], train['PRICE']
X_test, y_test = test[FEATURE_COLS], test['PRICE']
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score
import xgboost as xgb

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0, n_jobs=-1),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    cv_r2 = cross_val_score(model, X_train, y_train, cv=5, scoring='r2').mean()
    results[name] = {
        "R²": round(r2_score(y_test, pred), 4),
        "RMSE": round(np.sqrt(mean_squared_error(y_test, pred)), 0),
        "MAE": round(mean_absolute_error(y_test, pred), 0),
        "CV-R²": round(cv_r2, 4),
    }

df_results = pd.DataFrame(results).T.sort_values("R²", ascending=False)
print(df_results.to_string())

In [ ]:
# Sauvegarder le tableau des résultats
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
tbl = ax.table(cellText=df_results.reset_index().values,
               colLabels=["Modèle","R²","RMSE","MAE","CV-R²"],
               loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.5)
plt.title("Comparaison des 5 modèles", fontsize=13, pad=10)
plt.tight_layout(); plt.show()